18/04/2026 Fiestas de la UAB, se vieron picos altos de la amplitud

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import random
from scipy.stats import zscore

CSV_PATH = '../../data/12062026all_boxes.csv'

cols = [
    'Hive name',
    'Time',
    'Temperature heart',
    'Humidity heart',
    'Frequency',
    'Volume',
    'Temperature scale',
    'Humidity scale',
    'Weight'
]

df = pd.read_csv(CSV_PATH, usecols=cols)

df['Time'] = pd.to_datetime(
    df['Time'],
    errors='coerce',
    infer_datetime_format=True
)

df = df.dropna(subset=['Time'])
df = df.sort_values('Time')
df['date'] = df['Time'].dt.date
df['hour'] = df['Time'].dt.hour

# ============================================================
# TARGET EVENT
# ============================================================

TARGET_DATE = pd.to_datetime('2026-04-18').date()

print('\nAvailable hives:\n')
print(df['Hive name'].unique())

MIN_SAMPLES_TARGET_DAY = 100

valid_hives = []

print('\nChecking hives with valid data on 18/04...\n')

for hive in df['Hive name'].unique():

    sub = df[
        (df['Hive name'] == hive) &
        (df['date'] == TARGET_DATE)
    ]

    n = len(sub)

    #print(f'{hive:20s} -> {n} samples')

    if n >= MIN_SAMPLES_TARGET_DAY:

        valid_hives.append(hive)

selected_hives = valid_hives[:7]

print('SELECTED HIVES')
print(selected_hives)

df = df[
    df['Hive name'].isin(selected_hives)
]

def investigate_box(box_df, hive_name):

    print(f'HIVE: {hive_name}')

    box_df = box_df.copy()

    daily = (
        box_df
        .groupby('date')
        .agg({

            'Weight': ['mean', 'std', 'min', 'max'],

            'Temperature scale': ['mean', 'std'],
            'Humidity scale': ['mean', 'std'],

            'Temperature heart': ['mean', 'std'],
            'Humidity heart': ['mean', 'std'],

            'Frequency': ['mean', 'std'],

            'Volume': ['mean', 'std']

        })
    )

    daily.columns = [
        '_'.join(c).replace(' ', '_')
        for c in daily.columns
    ]

    daily = daily.reset_index()

    daily['weight_amplitude'] = (
        daily['Weight_max'] -
        daily['Weight_min']
    )

    target_row = daily[
        daily['date'] == TARGET_DATE
    ]

    if len(target_row) == 0:

        print('No data on 18/04')
        return None

    target_row = target_row.iloc[0]

    baseline = daily[
        daily['date'] != TARGET_DATE
    ]

    metrics = [

        'weight_amplitude',

        'Weight_std',

        'Frequency_mean',
        'Frequency_std',

        'Volume_mean',
        'Volume_std',

        'Temperature_scale_std',
        'Humidity_scale_std',

        'Temperature_heart_std',
        'Humidity_heart_std'
    ]

    results = []

    print('\nANOMALY Z-SCORES\n')

    for metric in metrics:

        if metric not in daily.columns:
            continue

        mu = baseline[metric].mean()
        sigma = baseline[metric].std()

        val = target_row[metric]

        if sigma == 0 or pd.isna(sigma):
            continue

        z = (val - mu) / sigma

        results.append({

            'Hive': hive_name,
            'metric': metric,
            'value_18_04': val,
            'baseline_mean': mu,
            'zscore': z

        })

        print(f'{metric:30s} '
              f'Value={val:8.2f} | '
              f'Baseline={mu:8.2f} | '
              f'Z={z:6.2f}')

    results = pd.DataFrame(results)

    # PLOT 1 — DAILY AMPLITUDE

    plt.figure(figsize=(14,5))

    plt.plot(
        daily['date'],
        daily['weight_amplitude'],
        linewidth=2
    )

    plt.axvline(
        TARGET_DATE,
        color='red',
        linestyle='--',
        linewidth=2,
        label='18/04'
    )

    plt.title(
        f'{hive_name} — Daily Weight Amplitude'
    )

    plt.ylabel('Amplitude')
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # PLOT 2 — WEIGHT AROUND EVENT

    start = TARGET_DATE - pd.Timedelta(days=2)
    end   = TARGET_DATE + pd.Timedelta(days=2)

    around = box_df[
        (box_df['date'] >= start) &
        (box_df['date'] <= end)
    ]

    plt.figure(figsize=(16,6))

    sns.lineplot(
        data=around,
        x='Time',
        y='Weight',
        hue='date',
        linewidth=1.5
    )

    plt.title(
        f'{hive_name} — Weight around 18/04'
    )

    plt.tight_layout()

    plt.show()

    # PLOT 3 — FREQUENCY

    if around['Frequency'].notna().sum() > 0:

        plt.figure(figsize=(16,5))

        sns.lineplot(
            data=around,
            x='Time',
            y='Frequency',
            hue='date'
        )

        plt.title(
            f'{hive_name} — Frequency around 18/04'
        )

        plt.tight_layout()

        plt.show()

    return results

all_results = []

for hive in selected_hives:

    box_df = df[
        df['Hive name'] == hive
    ]

    # remove hives with almost constant weight
    std_weight = box_df['Weight'].std()

    if std_weight < 0.1:
        continue

    res = investigate_box(box_df, hive)

    if res is not None:

        all_results.append(res)

if len(all_results) > 0:

    final = pd.concat(all_results)

    print('GLOBAL ANOMALY RANKING')

    print(
        final
        .sort_values('zscore', ascending=False)
        .head(30)
    )

# HEATMAP OF ANOMALIES

pivot = final.pivot_table(

    index='Hive',
    columns='metric',
    values='zscore'

)
plt.figure(figsize=(14,8))
sns.heatmap(

    pivot,
    annot=True,
    cmap='RdBu_r',
    center=0
)

plt.title(
    '18/04 anomaly intensity by hive and metric\n(Z-score vs normal behavior)'
)

plt.tight_layout()
plt.show()

In [ ]:

THRESHOLD = 700

extreme = df[
    df['Volume'] > THRESHOLD
].copy()

print('\n==============================')
print('EXTREME VOLUME EVENTS')
print('==============================')

print(f'Total events: {len(extreme)}')

events_by_day = (
    extreme
    .groupby('date')
    .size()
    .sort_values(ascending=False)
)

print('\nTop days:\n')

print(events_by_day.head(20))

events_by_hive = (
    extreme
    .groupby('Hive name')
    .size()
    .sort_values(ascending=False)
)

print('\nBy hive:\n')

print(events_by_hive)

plt.figure(figsize=(12,6))

plt.hist(
    df['Volume'].dropna(),
    bins=200
)

plt.axvline(
    819.1,
    color='red',
    linestyle='--',
    linewidth=2,
    label='Suspicious saturation value'
)

plt.xlim(0, 850)

plt.title('Volume distribution')

plt.xlabel('Volume')

plt.ylabel('Count')

plt.legend()

plt.tight_layout()

plt.show()

extreme['hour'] = extreme['Time'].dt.hour

plt.figure(figsize=(12,5))

sns.countplot(
    data=extreme,
    x='hour'
)

plt.title('Extreme acoustic events by hour')

plt.xlabel('Hour')

plt.ylabel('Count')

plt.tight_layout()

plt.show()

timeline = (
    extreme
    .groupby([
        'date',
        'Hive name'
    ])
    .size()
    .reset_index(name='count')
)

plt.figure(figsize=(16,6))

sns.scatterplot(
    data=timeline,
    x='date',
    y='Hive name',
    size='count',
    hue='count',
    sizes=(20,400)
)

plt.title('Timeline of extreme acoustic events')

plt.tight_layout()

plt.show()

df['extreme_audio'] = df['Volume'] > THRESHOLD

compare = (
    df
    .groupby('extreme_audio')[
        [
            'Frequency',
            'Weight',
            'Temperature scale',
            'Humidity scale'
        ]
    ]
    .mean()
)

print('\n==============================')
print('NORMAL VS EXTREME AUDIO')
print('==============================')

print(compare)
print('\n==============================')
print('MOST COMMON VOLUME VALUES')
print('==============================')

print(
    df['Volume']
    .round(1)
    .value_counts()
    .head(20)
)

sat = df[
    df['Volume'] >= 819
]

print('\n==============================')
print('SATURATION ANALYSIS')
print('==============================')

print(f'Rows at saturation: {len(sat)}')

print('\nDays with saturation:\n')

print(
    sat['date']
    .value_counts()
    .head(20)
)

print('\nHives with saturation:\n')

print(
    sat['Hive name']
    .value_counts()
)

if len(sat) > 0:

    print('\n================================================')
    print('INTERPRETATION')
    print('================================================')

    print("""
Possible sensor saturation detected.

Evidence:
- repeated exact value 819.1
- synchronized across hives
- concentrated on specific dates
- no equivalent biological response in weight

This strongly suggests:
external acoustic contamination OR sensor clipping.
""")

In [ ]:
# DID THE BEES STOP FORAGING DURING THE UAB EVENT?
from scipy.stats import ttest_ind
from scipy.stats import linregress
TARGET_DATE = pd.to_datetime('2026-04-18').date()
START_HOUR = 8
END_HOUR   = 20

day_df = df[
    (df['hour'] >= START_HOUR) &
    (df['hour'] <= END_HOUR)
].copy()

results = []
selected_hives = [1,2,3,4,6,7,8]

for hive in selected_hives:

    hive_df = day_df[
        day_df['Hive name'] == hive
    ].copy()

    for date, d in hive_df.groupby('date'):

        d = d.sort_values('Time')

        if len(d) < 10:
            continue

        amplitude = (
            d['Weight'].max() -
            d['Weight'].min()
        )

        morning = d[
            (d['hour'] >= 8) &
            (d['hour'] <= 12)
        ]

        slope = np.nan

        if len(morning) > 5:

            x = (
                morning['Time'] -
                morning['Time'].min()
            ).dt.total_seconds() / 3600

            y = morning['Weight']

            slope = linregress(x, y).slope

        midday = d[
            (d['hour'] >= 12) &
            (d['hour'] <= 14)
        ]

        midday_std = midday['Weight'].std()

        results.append({

            'Hive': hive,
            'date': date,

            'amplitude': amplitude,
            'morning_slope': slope,
            'midday_std': midday_std,

            'is_event': date == TARGET_DATE

        })

res = pd.DataFrame(results)

event = res[
    res['is_event']
]

normal = res[
    ~res['is_event']
]

print('FORAGING DYNAMICS')

metrics = [
    'amplitude',
    'morning_slope',
    'midday_std'
]

for m in metrics:

    e = event[m].dropna()
    n = normal[m].dropna()

    if len(e) == 0 or len(n) == 0:
        continue

    t, p = ttest_ind(e, n, equal_var=False)

    print(f'\n{m}')

    print(f'Event mean:  {e.mean():.3f}')
    print(f'Normal mean: {n.mean():.3f}')

    print(f'Difference:  {e.mean()-n.mean():.3f}')

    print(f'p-value:     {p:.5f}')

fig, axes = plt.subplots(1,3, figsize=(18,5))

for i, metric in enumerate(metrics):

    sns.boxplot(
        data=res,
        x='is_event',
        y=metric,
        ax=axes[i]
    )

    axes[i].set_xticklabels([
        'Normal days',
        '18/04 event'
    ])

    axes[i].set_title(metric)

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(
    len(selected_hives),
    1,
    figsize=(16, 3*len(selected_hives)),
    sharex=True
)

for i, hive in enumerate(selected_hives):

    hive_df = day_df[
        day_df['Hive name'] == hive
    ]

    around = hive_df[
        hive_df['date'].isin([
            TARGET_DATE - pd.Timedelta(days=1),
            TARGET_DATE,
            TARGET_DATE + pd.Timedelta(days=1)
        ])
    ]

    sns.lineplot(
        data=around,
        x='Time',
        y='Weight',
        hue='date',
        ax=axes[i]
    )

    axes[i].set_title(f'Hive {hive}')

plt.tight_layout()
plt.show()

print('INTERPRETATION')

print("""
If 18/04 shows:

- lower weight amplitude
- flatter morning slope
- reduced midday variability

then this supports:

REDUCED FORAGING ACTIVITY

potentially caused by:

EXTERNAL ACOUSTIC DISTURBANCE
during the UAB festivities.
""")

In [ ]:
# CROSS-CORRELATION ANALYSIS
# DID THE AUDIO EVENT ALTER COLONY DYNAMICS?

MAX_LAG = 24

def compute_crosscorr(x, y, max_lag=24):

    x = pd.Series(x).interpolate().fillna(method='bfill')
    y = pd.Series(y).interpolate().fillna(method='bfill')

    x = (x - x.mean()) / x.std()
    y = (y - y.mean()) / y.std()

    corrs = []

    lags = range(-max_lag, max_lag+1)

    for lag in lags:

        if lag < 0:
            corr = x[:lag].corr(y[-lag:])

        elif lag > 0:
            corr = x[lag:].corr(y[:-lag])

        else:
            corr = x.corr(y)

        corrs.append(corr)

    return lags, corrs

all_results = []

fig, axes = plt.subplots(
    len(selected_hives),
    2,
    figsize=(14, 4*len(selected_hives))
)

for i, hive in enumerate(selected_hives):

    hive_df = df[
        (df['Hive name'] == hive) &
        (df['date'] == TARGET_DATE)
    ].copy()

    hive_df = (
        hive_df
        .set_index('Time')
        .resample('5min')
        .mean(numeric_only=True)
        .reset_index()
    )

    if hive_df['Weight'].notna().sum() > 20:

        lags, corrs = compute_crosscorr(
            hive_df['Volume'],
            hive_df['Weight'],
            MAX_LAG
        )

        axes[i,0].plot(lags, corrs)

        axes[i,0].axvline(0, color='red')

        axes[i,0].set_title(
            f'Hive {hive}\nVolume ↔ Weight'
        )

        if not np.all(np.isnan(corrs)):
            best_lag = lags[np.nanargmax(np.abs(corrs))]
            best_corr = np.nanmax(np.abs(corrs))

            all_results.append({

                'Hive': hive,
                'pair': 'Volume-Weight',
                'best_lag': best_lag,
                'best_corr': best_corr

            })

    if hive_df['Temperature scale'].notna().sum() > 20:

        lags, corrs = compute_crosscorr(
            hive_df['Volume'],
            hive_df['Temperature scale'],
            MAX_LAG
        )

        axes[i,1].plot(lags, corrs)

        axes[i,1].axvline(0, color='red')

        axes[i,1].set_title(
            f'Hive {hive}\nVolume ↔ Temp'
        )

        if not np.all(np.isnan(corrs)):
            best_lag = lags[np.nanargmax(np.abs(corrs))]
            best_corr = np.nanmax(np.abs(corrs))

            all_results.append({

                'Hive': hive,
                'pair': 'Volume-Temp',
                'best_lag': best_lag,
                'best_corr': best_corr

            })

plt.tight_layout()
plt.show()

results = pd.DataFrame(all_results)

print('CROSS-CORRELATION RESULTS')

print(results)

print('INTERPRETATION')
print('======================================')

print("""
If strong lagged correlations exist:

Volume spike
→ delayed response in Weight/Temperature

then:
possible biological reaction.

If correlations are weak/random:

likely external acoustic contamination
or sensor artifact only.
""")